# **Prova Estrazione Highlights**

In questa prova utilizzo il dataset Learning From The Worst la cui definizione usata è:

"‘Hate’ is defined as “abusive speech targeting specific group characteristics, such as ethnic origin, religion, gender, or sexual orientation."

Le componenti sono Abuse, Ethnicity, Gender, Religion, Sexual Orientation.

I modelli che utilizzo per le estrazioni sono Llama-3.1-8b-Instruct e Qwen2.5-7B-Instruct

# HF login e imports

In [1]:
!pip install -U huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 668.2/668.2 kB 15.5 MB/s eta 0:00:00
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 1.11.0
    Uninstalling huggingface_hub-1.11.0:
      Successfully uninstalled huggingface_hub-1.11.0


In [ ]:
!hf auth login --token hf_**************

The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `hf`CLI if you want to set the git credential as well.
Token is valid (permission: read).
The token `NLP2` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
The current active token is: `NLP2`


In [3]:
!pip install --upgrade transformers accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 56.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 10.9 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [4]:
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import numpy as np
import json
import random
from IPython.display import display, HTML

# Data Processing

In [5]:
!gdown 1pktiqIn_fRDcQY7FkPT3To6nPwrujF3n

Downloading...
From: https://drive.google.com/uc?id=1pktiqIn_fRDcQY7FkPT3To6nPwrujF3n
To: /content/Dynamically Generated Hate Dataset v0.2.3.csv
100% 9.76M/9.76M [00:00<00:00, 43.1MB/s]


In [6]:
df=pd.read_csv('/content/Dynamically Generated Hate Dataset v0.2.3.csv')
df.head()

,Unnamed: 0,acl.id,X1,text,label,type,target,level,split,round.base,annotator,round,acl.id.matched
0,1,acl1,1,dalits are lowlives,hate,notgiven,notgiven,original,train,1,GNZuCtwed3,1,NaN
1,2,acl10,2,gay people are a burden to our society,hate,notgiven,notgiven,original,train,1,GNZuCtwed3,1,NaN
2,3,acl100,3,Arabs are not welcome,hate,notgiven,notgiven,original,train,1,vDe7GN0NrL,1,NaN
3,4,acl1000,4,I'm not saying we should actually eliminate he...,hate,notgiven,notgiven,original,train,1,oemYWm1Tjg,1,NaN
4,5,acl10000,5,bananas are for black people,hate,notgiven,notgiven,original,test,1,QiOKkCi7F8,1,NaN


In [7]:
hate_rows = df[df['label'] == 'hate']
hate_rows = hate_rows[(hate_rows['type'] != 'notgiven') & (hate_rows['target'] != 'notgiven')]
hate_rows.head()

,Unnamed: 0,acl.id,X1,text,label,type,target,level,split,round.base,annotator,round,acl.id.matched
1256,1291,acl11159,1291,There is way too many m0ng0l0id kids at my sch...,hate,derogation,dis,original,train,2,TrRF46JWfP,2a,acl19858
1259,1294,acl11161,1294,the worst of them are the taaaaaaaaaards becau...,hate,derogation,dis,original,train,2,TrRF46JWfP,2a,acl11162
1261,1296,acl11163,1296,it is not niiiice to be around reeetaaaardss,hate,derogation,dis,original,train,2,TrRF46JWfP,2a,acl11164
1263,1298,acl11165,1298,travellers have a certain aroma,hate,animosity,trav,original,train,2,CAgNlUizNm,2a,acl11166
1265,1300,acl11167,1300,Can you smell these dirty muzzies? It's a brut...,hate,derogation,mus,original,train,2,CAgNlUizNm,2a,acl11168


In [8]:
print("Statistiche sul dataset filtrato:")
print(f"Totale righe hate: {len(hate_rows)}")
print()
print("Distribuzione tipi:")
print(hate_rows['type'].value_counts(dropna=False))
print()
print("Distribuzione target:")
print(hate_rows['target'].value_counts(dropna=False))
print()
print("Tabella incrociata type x target:")
display(pd.crosstab(hate_rows['type'], hate_rows['target'], margins=True))


Statistiche sul dataset filtrato (hate rows):
Totale righe hate: 15065

Distribuzione tipi:
type
derogation        9907
animosity         3439
dehumanization     906
threatening        606
support            207
Name: count, dtype: int64

Distribuzione target:
target
wom                     2035
bla                     1961
jew                     1096
mus                     1002
trans                    792
                        ... 
trans, lgbtq               1
jew, other.religion        1
asi, immig, asi.chin       1
wom, asi.pak               1
trav, wom                  1
Name: count, Length: 408, dtype: int64

Tabella incrociata type x target:


target,african,"african, asylum",arab,"arab, african","arab, asi.chin","arab, ref","arab, ref, african",asi,"asi, asi.chin","asi, asi.chin, asi.wom",...,"wom, jew, gendermin","wom, jew, mixed.race","wom, jew, mixed.race, non.white","wom, jew, non.white","wom, mixed.race","wom, mus.wom","wom, non.white","wom, pol","wom, wc",All
type,,,,,,,,,,,,,,,,,,,,,
animosity,22,0,59,1,1,4,0,38,0,0,...,0,0,0,1,2,1,2,0,1,3439
dehumanization,32,1,13,1,0,1,0,9,1,1,...,0,0,0,0,0,0,0,0,0,906
derogation,68,1,250,2,1,4,1,88,4,0,...,1,1,1,2,2,3,3,1,1,9901
support,0,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,205
threatening,1,0,15,0,0,0,0,4,0,0,...,0,0,0,0,0,0,0,0,0,606
All,123,2,338,4,2,9,1,139,5,1,...,1,1,1,3,4,4,5,1,2,15057


In [9]:
n_per_type = 20

sampled_rows = hate_rows.groupby('type', group_keys=False).apply(
    lambda group: group.sample(min(len(group), n_per_type), random_state=42)
).reset_index(drop=True)
print(sampled_rows['type'].value_counts())

sampled_rows = sampled_rows[['text', 'type', 'target']]
sampled_rows.to_csv('sampled_rows_LFTW.csv', index=False)
sampled_rows.head()

type
animosity         20
dehumanization    20
derogation        20
support           20
threatening       20
Name: count, dtype: int64


/tmp/ipykernel_2363/3012567020.py:3: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  sampled_rows = hate_rows.groupby('type', group_keys=False).apply(


,text,type,target
0,"Burning TikTok isn't a bad thing, especially i...",animosity,asi.east
1,"Donald Trump, please make gay bullying accepta...",animosity,gay
2,In particular what I would say because of wher...,animosity,mixed.race
3,Yeah I have a fugly yt friend who had to resor...,animosity,bla
4,Muslim women are just big letterboxes,animosity,mus.wom


In [12]:
print("Distribuzione target:")
print(sampled_rows['target'].value_counts(dropna=False))

Distribuzione target:
target
bla                    15
wom                    10
jew                     8
nazis                   7
gay                     6
immig                   6
trans                   4
mus                     4
hitler                  3
asi.south               3
notargetrecorded        3
asi.east                2
dis                     2
gay.man                 2
asi.chin                2
asi.wom                 2
asi                     2
mixed.race              1
ref                     1
bla.wom, jew            1
arab                    1
immig, ref, asylum      1
mus.wom                 1
gendermin, bis          1
trav                    1
mus, arab               1
indig                   1
wc                      1
other.glorification     1
NaN                     1
non.white.wom           1
eastern.europe          1
immig, hispanic         1
asylum                  1
jew, gay                1
gendermin               1
Name: count, dtype: int64


In [23]:
count_by_type = hate_rows['type'].value_counts(dropna=False).rename_axis('type').reset_index(name='count')
count_by_target = hate_rows['target'].value_counts(dropna=False).rename_axis('target').reset_index(name='count')
sampled_rows_count_by_target = sampled_rows['target'].value_counts(dropna=False).rename_axis('target').reset_index(name='count')

summary = {
    'total_hate_rows': len(hate_rows),
    'types': count_by_type.to_dict(orient='records'),
    'targets': count_by_target.to_dict(orient='records'),
    'sampled_rows_targets': sampled_rows_count_by_target.to_dict(orient='records')
}

with open('LFTW_stats.json', 'w', encoding='utf-8') as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)


In [11]:
texts=sampled_rows['text'].sample(100).reset_index(drop=True)
texts

,text
0,I believe that identifying as a black man inte...
1,"Kill the mussies, so that islamism can die thi..."
2,"Bro here, someone posted a selfie with ""Fuck T..."
3,About 12 days and no charges? There is no way ...
4,Muslim women are just big letterboxes
...,...
95,"Why are Indians being bellends pls, what is th..."
96,"The ""angrier world"" will slaughter most of dra..."
97,"I was really impressed with Paris, it was all ..."
98,The Natsees denounced the Bolshevik criminals ...


# Load Model

In [14]:
quantization_config = BitsAndBytesConfig(load_in_4bit=True)

model_name_1="meta-llama/Llama-3.1-8B-Instruct"

tokenizer_1 = AutoTokenizer.from_pretrained(model_name_1)
model_1 = AutoModelForCausalLM.from_pretrained(
    model_name_1,
    device_map="auto",
    quantization_config=quantization_config
  )
#7 minuti

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

In [15]:
quantization_config = BitsAndBytesConfig(load_in_4bit=True)

model_name_2="Qwen/Qwen2.5-7B-Instruct"

tokenizer_2 = AutoTokenizer.from_pretrained(model_name_2)
model_2 = AutoModelForCausalLM.from_pretrained(
    model_name_2,
    device_map="auto",
    quantization_config=quantization_config
  )
#8 minuti

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/27.8k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

# Prompt

In [ ]:
prompt = [
    {
        "role": "system",
        "content": "You are a data annotator specialized in hate speech detection."
    },
    {
        "role": "user",
        "content": """Read the provided TEXT and extract the exact substring that correspond to the following Hate Speech components.
        If a component is not present in the text, you must return the string "absent".

        Below you find the components to identify in the given text:

        - Abuse
        - Ethnicity
        - Gender
        - Religion
        - Sexual Orientation

        RULES:
        1- You must extract exact quotes from the text. Do not paraphrase or change any words.
        2- Do not output any conversational text, greetings, introductions, or explanations.
        3- Your response must be only the raw json object. Do not wrap the json in markdown formatting.

        OUTPUT FORMAT:
        {
            "Abuse": "extracted substring or absent",
            "Ethnicity": "extracted substring or absent",
            "Gender": "extracted substring or absent",
            "Religion": "extracted substring or absent",
            "Sexual Orientation": "extracted substring or absent"
        }


        TEXT: "{text}"

        ANSWER:

        """
    }
]

In [16]:
prompt = [
    {
        "role": "system",
        "content": "You are a data annotator specialized in hate speech detection."
    },
    {
        "role": "user",
        "content": """Read the provided TEXT and extract the exact substring that correspond to the following Hate Speech components.
        If a component is not present in the text, you must return the string "absent".

        Below you find the components to identify in the given text:

        - Abuse:  Content which involves improper treatment of a person or group, often to unfairly or improperly gain benefit.
        - Derogation: Content which explicitly attacks, demonizes, demeans or insults a group
        - Animosity: Content which expresses abuse against a group in an implicit or subtle manner.
        - Threatening language: Content which expresses intention to, support for, or encourages inflicting harm on a group, or identified members of the group.
        - Support for hateful entities: Content which explicitly glorifies, justifies or supports hateful actions, events, organizations, tropes and individuals.
        - Dehumanization Content which ‘perceiv[es] or treat[s] people as less than human’
        - Ethnicity: Any reference to a person's or group's ethnic origin.
        - Gender: Any reference to a person's or group's gender identity or biological sex.
        - Religion: Any reference to a person's or group's religious beliefs, practices, or affiliation.
        - Sexual Orientation: Any reference to a person's or group's sexual orientation.

        OUTPUT FORMAT:
        {
            "Abuse": "extracted substring or absent",
            "Derogation": "extracted substring or absent",
            "Animosity": "extracted substring or absent",
            "Threatening language": "extracted substring or absent",
            "Support for hateful entities": "extracted substring or absent",
            "Dehumanization": "extracted substring or absent",
            "Ethnicity": "extracted substring or absent",
            "Gender": "extracted substring or absent",
            "Religion": "extracted substring or absent",
            "Sexual Orientation": "extracted substring or absent"
        }

        RULES:
        1- You must extract exact quotes from the text. Do not paraphrase or change any words.
        2- Do not output any conversational text, greetings, introductions, or explanations.
        3- Your response must be ONLY the raw json object. Do not wrap the json in markdown formatting.

        TEXT: "{text}"

        ANSWER:

        """
    }
]

In [17]:
def prepare_prompts(texts, prompt_template, tokenizer):
  """
    This function format input text samples into instructions prompts.

    Inputs:
      texts: input texts to classify via prompting
      prompt_template: the prompt template provided in this assignment
      tokenizer: the transformers Tokenizer object instance associated
      with the chosen model card

    Outputs:
      input texts to classify in the form of instruction prompts
  """
  formatted_prompts=[]

  for text in texts:
    prompt_copy = [
            {"role": prompt_template[0]["role"], "content": prompt_template[0]["content"]},
            {
                "role": prompt_template[1]["role"],
                "content": prompt_template[1]["content"].replace("{text}", text)
            }
        ]


    formatted_prompt = tokenizer.apply_chat_template(
      prompt_copy,
      add_generation_prompt=True,
      tokenize=True,
      return_dict=True,
      return_tensors="pt",
    )

    formatted_prompts.append(formatted_prompt)


  return formatted_prompts

# Inference

In [27]:
def generate_responses(model, prompt_examples, tokenizer, verbose=False):
  """
    This function implements the inference loop for a LLM model.
    Given a set of examples, the model is tasked to generate
    a response.

    Inputs:
      model: LLM model instance for prompting
      prompt_examples: pre-processed text samples

    Outputs:
      generated responses
  """
  outputs=[]
  for index, prompt_example in enumerate(prompt_examples):
    if verbose and index % 10 == 0:
      print(f"generating response n.{index}...")
    prompt_example=prompt_example.to(model.device)
    output = model.generate(**prompt_example, max_new_tokens=500,pad_token_id=tokenizer.eos_token_id)
    output = tokenizer.decode(output[0][prompt_example["input_ids"].shape[-1]:], skip_special_tokens=True)
    outputs.append(output)

  return outputs

In [28]:
outputs1 = generate_responses(model_1, prepare_prompts(texts, prompt, tokenizer_1), tokenizer_1, verbose=True)

generating response n.0...
generating response n.10...
generating response n.20...
generating response n.30...
generating response n.40...
generating response n.50...
generating response n.60...
generating response n.70...
generating response n.80...
generating response n.90...


In [29]:
parsed_outputs = []
json_errors = []
for output in outputs1:
    try:
        parsed_outputs.append(json.loads(output))
    except json.JSONDecodeError:
        json_errors.append(output)

with open("results_LFTW_1.json", "w", encoding="utf-8") as f:
    json.dump(parsed_outputs, f, ensure_ascii=False, indent=2)

with open("json_errors_LFTW_1.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(json_errors))
    f.write("\n")
    f.write("--"*20)

In [30]:
outputs2 = generate_responses(model_2, prepare_prompts(texts, prompt, tokenizer_2), tokenizer_2, verbose=True)

generating response n.0...
generating response n.10...
generating response n.20...
generating response n.30...
generating response n.40...
generating response n.50...
generating response n.60...
generating response n.70...
generating response n.80...
generating response n.90...


In [31]:
parsed_outputs2 = []
json_errors2 = []
for output in outputs2:
    try:
        parsed_outputs2.append(json.loads(output))
    except json.JSONDecodeError:
        json_errors2.append(output)

with open("results_LFTW_2.json", "w", encoding="utf-8") as f:
    json.dump(parsed_outputs2, f, ensure_ascii=False, indent=2)

with open("json_errors_LFTW_2.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(json_errors2))
    f.write("\n")
    f.write("--"*20)

# Extraction Results

In [ ]:
def highlights(texts, outputs):

  """
      This function displays the given text with the model extracted highlights.

  """

  colori = {
        "Abuse": "#FFB3B3",
        "Ethnicity": "#B3D9FF",
        "Gender": "#E6B3FF",
        "Religion": "#B3FFB3",
        "Sexual Orientation": "#FFD9B3"
      }

  testo_colorato = texts

  for categoria, estratto in outputs.items():
      if estratto and estratto.lower() != "null" and estratto in testo_colorato:
          colore = colori.get(categoria, "#DDDDDD")
          tag_html = f"<mark style='background-color: {colore}; padding: 2px 4px; border-radius: 4px;'>{estratto} <span style='font-size: 0.7em; font-weight: bold;'>[{categoria}]</span></mark>"
          testo_colorato = testo_colorato.replace(estratto, tag_html)

  display(HTML(f"<p style='line-height: 1.8; font-size: 16px;'>{testo_colorato}</p>"))


In [ ]:
for i in range(len(outputs1)):
    parsed_output = {}
    try:
        parsed_output = json.loads(outputs1[i])
    except json.JSONDecodeError as e:
        print(f"JSONDecodeError: {e} for text: {texts[i]}, model output: {outputs1[i]}")

    highlights(texts[i], parsed_output)

JSONDecodeError: Expecting ':' delimiter: line 4 column 27 (char 145) for text: Most Americans don’t know it but state-sanctioned “gay marriage” in America was built on a massive lie emanating from the cunningly deceitful gay movement: that “being gay” (or even trans, or bisexual, or “pansexual” ] is “who you are,” innate and unchangeable, like race., model output: {
  "Abuse": "built on a massive lie emanating from the cunningly deceitful gay movement",
  "Ethnicity": "Americans",
  "Gender": "gay", "trans", "bisexual", "pansexual",
  "Religion": "absent",
  "Sexual Orientation": "gay", "trans", "bisexual", "pansexual"
}


JSONDecodeError: Expecting ':' delimiter: line 3 column 41 (char 75) for text: I believe bangladeshis are no better than Indians. In SEA, they are behaving well and generally more clean. It is a surprise they get frequently called with derogatory remarks, model output: {
  "Abuse": "derogatory remarks",
  "Ethnicity": "bangladeshis", "Indians",
  "Gender": "absent",
  "Religion": "absent",
  "Sexual Orientation": "absent"
}


JSONDecodeError: Expecting ':' delimiter: line 3 column 43 (char 71) for text: Everyone knows that Christians are weak cowards. That's why these muhhamedans act like this. The only strong spiritual movement is the third position nationalist., model output: {
  "Abuse": "weak cowards",
  "Ethnicity": "Christians", "muhhamedans",
  "Gender": "absent",
  "Religion": "Christians", "muhhamedans", "spiritual",
  "Sexual Orientation": "absent"
}


In [ ]:
for i in range(len(outputs2)):
    parsed_output = {}
    try:
        parsed_output = json.loads(outputs2[i])
    except json.JSONDecodeError as e:
        print(f"JSONDecodeError: {e} for text: {texts[i]}, model output: {outputs2[i]}")

    highlights(texts[i], parsed_output)